In [ ]:
# Import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import joblib


# Gunakan dataset hasil clustering yang memiliki fitur Target
# Silakan gunakan dataset data_clustering jika tidak menerapkan Interpretasi Hasil Clustering [Advanced]
# Silakan gunakan dataset data_clustering_inverse jika menerapkan Interpretasi Hasil Clustering [Advanced]

# Gunakan dataset hasil clustering yang memiliki fitur Target
df = pd.read_csv("data_clustering_inverse.csv")

# Tampilkan 5 baris pertama dengan function head
print(df.head())

# Memilih Kategori Data
categorical_cols = list(df.select_dtypes(include=['object']).columns)

# Menggunakan 'pd.get_dummies' untuk melakukan OneHotEncoding
df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)

# Tampilkan 5 baris pertama untuk memverifikasi hasilnya
print(df_encoded.head())



# Menggunakan train_test_split() untuk melakukan pembagian dataset.
# Buat 'X' dengan menghapus 'Target' dari 'df_encoded' dan gunakan 'axis=1' untuk menandakan drop kolom.
X = df_encoded.drop('Target', axis=1)

# Buat 'y' dengan HANYA memilih kolom 'Target'.
y = df_encoded['Target']

# Panggil fungsi untuk membagi data.
#  - Gunakan 'stratify=y' agar proporsi kelas di train/test set sama.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.3,
    random_state = 42,
    stratify = y
)

print("Jumlah data total: ",len(X))
print("Jumlah data latih: ",len(X_train))
print("Jumlah data test: ",len(X_test))


# Buatlah model klasifikasi menggunakan Decision Tree
decision_tree_model = DecisionTreeClassifier(random_state=42)

# 2. Latih (fit) model dengan data training (X_train dan y_train)
decision_tree_model.fit(X_train, y_train)

# Menyimpan Model
joblib.dump(decision_tree_model, 'decision_tree_model.h5')

# Melatih model menggunakan algoritma klasifikasi scikit-learn selain Decision Tree. (Contoh: RandomForestClassifier)
new_model = RandomForestClassifier(random_state=42)

# Latih (fit) model dengan data training (X_train dan y_train)
new_model.fit(X_train, y_train)


# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma yang sudah dibuat.

# Buat prediksi pada data 'X_test' menggunakan kedua model
y_pred_dt = decision_tree_model.predict(X_test)
y_pred_new = new_model.predict(X_test)

# Tampilkan classification_report untuk Decision Tree
print("Decision Tree Performance")
print(classification_report(y_test, y_pred_dt))

print("="*50)

# Tampilkan classification_report untuk New Model
print("New Model Performance")
print(classification_report(y_test, y_pred_new))


# Menyimpan Model Selain Decision Tree(Random Forest)

joblib.dump(new_model, 'explore_random_forest_classification.h5')


# Lakukan dalam satu cell ini saja.


# Tentukan Hyperparameter yang akan di-tuning
params = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Buat (instantiate) objek model baru
# Buat (instantiate) objek dari algoritma tuning
#  - 'estimator': Model yang akan di-tuning
#  - 'params': Hyperparameter yang sudah kita tentukan
new_model_tuned = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42),
    param_grid= params,
    cv = 5,
    scoring = 'accuracy'
)


# Latih objek model dengan data training (X_train dan y_train)
new_model_tuned.fit(X_train, y_train)


# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.


# Buat prediksi pada 'X_test' Gunakan model yang sudah di-tuning
y_pred_tuning = new_model_tuned.predict(X_test)

# Tampilkan classification_report untuk model yang sudah di-tuning
print("Tuned Model Performance")
print(classification_report(y_test, y_pred_tuning))

# Menyimpan Model hasil tuning
joblib.dump(new_model_tuned, 'tuning_classification.h5')

